# M3 · Keypoints de Ruedas y Conteo de Ejes (`ejes.pt`) — Dataset Combinado v2
Entrena **YOLOv8 Pose (`yolov8n-pose.pt`)** sobre el dataset combinado `/entrenamiento/datasets/m3_adaptado_v2/` generado por `entrenamiento/convertir_m3_v2.py`:
- Fuente 1: `/coco-m3/` (Vehicle Wheel Pose — Roboflow).
- Fuente 2: `/coco-m3-extra/truck-dataset/` (**Truck Image Dataset v2.0.0**, Universidad de São Paulo, Zenodo 7695824, licencia **CC BY 4.0**).

**Configuración:**
- Clase única: `0: vehiculo`
- Esquema uniforme: `kpt_shape: [7, 3]` (`eje_1`..`eje_7`, orden frente -> atrás)
- Antes de ejecutar en Google Colab: selecciona **GPU**, comprime `entrenamiento/datasets/m3_adaptado_v2/` como `m3_adaptado_v2.zip` y súbelo a `MyDrive/m3_adaptado_v2.zip`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics
import torch, ultralytics
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount("/content/drive")
ZIP_PATH = Path("/content/drive/MyDrive/m3_adaptado_v2.zip")
DATASET_DIR = Path("/content/m3_adaptado_v2")

if not DATASET_DIR.exists():
    assert ZIP_PATH.is_file(), f"Sube primero m3_adaptado_v2.zip a {ZIP_PATH}"
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")

yaml_path = DATASET_DIR / "data.yaml"
if not yaml_path.is_file():
    candidatos = list(Path("/content").rglob("data.yaml"))
    assert candidatos, "No se encontró data.yaml tras descomprimir"
    yaml_path = candidatos[0]
    DATASET_DIR = yaml_path.parent

contenido = yaml_path.read_text(encoding="utf-8").replace("path: .", f"path: {DATASET_DIR}")
yaml_path.write_text(contenido, encoding="utf-8")
print(yaml_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO

m = YOLO("yolov8n-pose.pt")
m.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/peaje",
    name="m3_v2",
    exist_ok=True,
    seed=42,
    fliplr=0.0,
)

best = "/content/peaje/m3_v2/weights/best.pt"
m = YOLO(best)
metrics = m.val(data=str(yaml_path), split="test")
print("mAP50 pose:", metrics.pose.map50)
print("mAP50-95 pose:", metrics.pose.map)

out_dir = Path("/content/drive/MyDrive/smart-peaje-evidencias/m3_v2")
out_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(best, "/content/drive/MyDrive/ejes.pt")
shutil.copy(best, out_dir / "ejes.pt")
for fname in ["results.csv", "results.png", "PosePR_curve.png", "PoseF1_curve.png"]:
    src = Path("/content/peaje/m3_v2") / fname
    if src.is_file():
        shutil.copy(src, out_dir / fname)

In [ ]:
# Evaluación de MAE y exactitud de conteo de ejes en test (desglosado por fuente y por número real de ejes)
from collections import defaultdict
import json
from pathlib import Path
import numpy as np

meta_path = DATASET_DIR / "metadatos_procedencia.json"
meta_por_img = {}
if meta_path.is_file():
    meta_raw = json.loads(meta_path.read_text(encoding="utf-8"))
    for item in meta_raw.get("imagenes_incluidas", []):
        if item.get("split") == "test":
            meta_por_img[item["archivo_imagen"]] = item

test_imgs = sorted((DATASET_DIR / "test" / "images").glob("*"))
test_lbls = DATASET_DIR / "test" / "labels"
UMBRAL_KPT = 0.5

errores_global = []
errores_por_fuente = defaultdict(list)
errores_por_eje = defaultdict(list)
exactos = 0
detalle = []

for img_path in test_imgs:
    lbl_path = test_lbls / f"{img_path.stem}.txt"
    if not lbl_path.is_file():
        continue
    lineas = [l.strip() for l in lbl_path.read_text().splitlines() if l.strip()]
    if len(lineas) != 1:
        continue
    valores = [float(x) for x in lineas[0].split()]
    kpts_real = valores[5:]
    ejes_visibles_gt = sum(1 for i in range(2, len(kpts_real), 3) if kpts_real[i] > 0)
    if ejes_visibles_gt < 2:
        continue

    meta_img = meta_por_img.get(img_path.name, {})
    fuente = meta_img.get("fuente", "desconocida")
    inst_meta = meta_img.get("instancias", [{}])[0]
    ejes_reales = int(inst_meta.get("ejes_reales") or ejes_visibles_gt)

    r = m.predict(str(img_path), conf=0.35, verbose=False)[0]
    ejes_pred = 0
    if r.keypoints is not None and len(r.keypoints) >= 1 and r.keypoints.conf is not None:
        best_k = int(r.boxes.conf.argmax()) if (r.boxes is not None and len(r.boxes) > 0) else 0
        ejes_pred = int((r.keypoints.conf[best_k] >= UMBRAL_KPT).sum())

    err = abs(ejes_pred - ejes_reales)
    errores_global.append(err)
    errores_por_fuente[fuente].append(err)
    errores_por_eje[str(ejes_reales)].append(err)
    if err == 0:
        exactos += 1
    detalle.append({
        "imagen": img_path.name,
        "fuente": fuente,
        "ejes_reales": ejes_reales,
        "ejes_predichos": ejes_pred,
        "error_abs": err,
    })

mae_global = float(np.mean(errores_global)) if errores_global else None
acc_exacta = (exactos / len(errores_global) * 100.0) if errores_global else None
resumen_eval = {
    "map50_pose": float(metrics.pose.map50),
    "map50_95_pose": float(metrics.pose.map),
    "mae_ejes_global": mae_global,
    "exactitud_conteo_exacto_pct": round(acc_exacta, 2) if acc_exacta is not None else None,
    "mae_por_fuente": {k: round(float(np.mean(v)), 4) for k, v in errores_por_fuente.items()},
    "mae_por_ejes_reales": {k: {"n": len(v), "mae": round(float(np.mean(v)), 4)} for k, v in sorted(errores_por_eje.items())},
    "umbral_kpt": UMBRAL_KPT,
    "total_test_evaluadas": len(errores_global),
}
print(json.dumps(resumen_eval, indent=2, ensure_ascii=False))
(out_dir / "metricas_m3_v2.json").write_text(
    json.dumps({**resumen_eval, "detalle": detalle}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)